
# Experiment v11c: COMMUTE in its multi-source form against targeted synthetic data

Builds on v10 (multiple donors) and adds the closest published method, **COMMUTE** (Gu, Lee & Duan, *J Biomed Inform* 2023), in its multi-source form:

1. each donor shares its trained logistic model **once**;
2. at the receiver, each donor model $k$ is **transferred separately**: $\mathrm{logit}\,p_k=\hat\beta_k^\top x+\delta_{0k}+\delta_k^\top z$, with penalty $\tfrac{\lambda_k}{2}\|\delta_k\|^2$ and $\lambda_k$ chosen by 5-fold CV on the local patients;
3. the receiver generates **synthetic covariates from its own patients** (sequential CART on the 200 local rows), labels them with the average of the calibrated donor models, and fits the final model on local + synthetic data (arm `COMMUTE_like`).

Arm `transfer_mix` is the single-transfer version from v11a, applied to the averaged donor score.

## Pre-registered hypotheses
- **K1:** wherever the framework predicts a gain ($\gamma=1$), `C_union` (donor synthetic data targeted at the receiver) beats `COMMUTE_like`. COMMUTE's covariates come only from 200 local patients, whereas the donors' pools supply thousands of targeted patients.
- **K2:** `COMMUTE_like` is at least as good as `transfer_mix`, because it calibrates each donor separately.
- **K3:** at $\gamma=0$, `COMMUTE_like` does not beat updated model sharing (its lower CI bound is ≤ 0).

(All v10 hypotheses M1–M4 and outputs are kept.)


---
# Experiment v10: several donor hospitals

All previous experiments used one large donor. In practice a small unit may learn from several hospitals, which may resemble it more or less.
This experiment keeps the total donor size fixed (10,000 patients) and compares three configurations:

| Configuration | Donors (share of patients above the BMI median) |
|---|---|
| `single` | one donor of 10,000, mirror of the receiver |
| `three_opposite` | three donors of 3,334, all mirror of the receiver |
| `three_diverse` | three donors of 3,334: one mirror, one balanced (50%), one like the receiver |

**Outcome:** semi-synthetic, as in v6/v8a, with γ = 0 (control) or γ = 1. The receiver has 90% or 70% of its patients above the median, 200 local patients, and 9,500 test patients. 20 repetitions.

**How each method uses several donors:**
- **Model sharing:** each donor publishes its logistic score; the receiver averages their logits, then applies closed testing (`share_mix_ct`).
- **Direct reweighting:** each donor refits its score with the weights returned by the receiver; the receiver averages the refitted logits (`IW_mix`). This needs one exchange per donor.
- **Targeted synthetic data:** the receiver pools all donors' synthetic pools, each labelled by its own donor's teacher, and targets the union with one density-ratio classifier (`C_union`). `B_union` is the same union without targeting.

**Predicted gain:** Γ_mix = E KL(h*_R ‖ averaged population donor scores), and its finite-sample version after closed testing on 200 patients.

## Pre-registered hypotheses
- **M1:** γ = 0 → no gain of `C_union` over `share_mix_ct`. γ = 1 → a significant gain wherever the predicted finite-sample gain is ≥ 2.
- **M2:** targeting the union of diverse donors has a larger effective sample size than targeting a single mirrored donor.
- **M3:** with diverse donors the predicted gain is smaller than with a single mirrored donor (one donor already resembles the receiver), and the observed gain is smaller accordingly.
- **M4:** with three smaller mirrored donors, reweighting suffers more from ratio estimation, so the advantage of targeted synthetic data over reweighting is larger than with one large donor.

In [1]:
%pip install -q sdv xgboost statsmodels ucimlrepo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Configuration

In [2]:
DATASET = "natality"          # "cdc_diabetes" (UCI 891) or "natality"
NAT_URL, NAT_READ, NAT_N = "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/DVS/natality/Nat2023us.zip", 0.02, 10**9
CSV_PATH, TARGET_COL, POSITIVE, DROP_COLS = "x.csv", None, None, []

SHIFT_VAR, AGE_VAR = "BMI", "Age"
GAMMAS   = [0.0, 1.0]             # 0 = control (no misspecification), 1 = strong misspecification
SKEWS    = [0.7, 0.9]             # receiver share above the BMI median
N_LOCAL  = 200
N_DONOR  = 10000
TEST_N   = 9500
N_SYN    = 10000                  # synthetic patients sent by the donor
POOL_MULT = 3                     # synthetic pool for targeting = POOL_MULT x N_DONOR
POP_N    = 15000 if DATASET == "cdc_diabetes" else 5000   # population samples for the oracles
W_CLIP_Q, CART_MIN_LEAF, ALPHA_CT = 0.99, 5, 0.05

QUICK  = False                     # True: 2 repetitions. False: 20 repetitions.
N_REPS = 2 if QUICK else 20
SEED0  = 2026
OUT_CSV = f"results_v11c_{DATASET}_{'quick' if QUICK else 'full'}.csv"
print(f"{DATASET} QUICK={QUICK} reps={N_REPS} -> {OUT_CSV}")

# donor configurations: (share above the BMI median, size). "opp" = mirror of the receiver, "mid" = balanced, "same" = like the receiver
CONFIGS = {"single":         [("opp", 10000)],
           "three_opposite": [("opp", 3334), ("opp", 3334), ("opp", 3334)],
           "three_diverse":  [("opp", 3334), ("mid", 3334), ("same", 3334)]}
CFG_NAMES = list(CONFIGS)
def dshare(kind, sk): return {"opp": 1 - sk, "mid": 0.5, "same": sk}[kind]


natality QUICK=False reps=20 -> results_v11c_natality_full.csv


## 2. Data and semi-synthetic outcome

In [3]:
import warnings, os, time, itertools
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

NAT_FIELDS = {   # 1-based positions from the NCHS 2023 natality user guide -> python slices
    "MAGER": (74, 76), "BMI": (282, 286), "RF_PDIAB": (312, 313), "RF_PHYPE": (314, 315),
    "RF_GHYPE": (315, 316), "RF_EHYPE": (316, 317), "RF_PPTERM": (317, 318), "RF_INFTR": (324, 325),
    "CIG_0": (252, 254), "PRIORLIVE": (170, 172), "PRIORDEAD": (172, 174), "MEDUC": (123, 124),
    "DPLURAL": (453, 454), "PAY_REC": (435, 436)}

def _zip_lines(zpath):
    # The CDC archive uses Deflate64, which Python's zipfile cannot read; stream it through the system unzip tool instead.
    import zipfile, subprocess, shutil
    try:
        with zipfile.ZipFile(zpath) as z:
            name = [n for n in z.namelist() if n.lower().endswith(".txt")][0]
            with z.open(name) as f:
                yield from f
        return
    except NotImplementedError:
        pass
    if shutil.which("unzip"):
        cmd = ["unzip", "-p", zpath]
    elif shutil.which("7z"):
        cmd = ["7z", "e", "-so", zpath]
    else:
        raise RuntimeError("Install unzip first:  !apt-get -qq install unzip")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, bufsize=1 << 20)
    yield from p.stdout
    if p.wait() != 0:
        raise RuntimeError(f"{cmd[0]} failed while reading {zpath}")

def load_natality():
    import urllib.request, zipfile, random
    cache = "natality2023_sample.csv"
    if not os.path.exists(cache):
        zpath = "Nat2023us.zip"
        if not os.path.exists(zpath):
            print("downloading the 2023 natality file (about 230 MB) ..."); urllib.request.urlretrieve(NAT_URL, zpath)
        rnd = random.Random(SEED0); rows = []
        for line in _zip_lines(zpath):
            if rnd.random() >= NAT_READ: continue
            s = line.decode("latin-1")
            rows.append([s[a:b] for a, b in NAT_FIELDS.values()])
        pd.DataFrame(rows, columns=list(NAT_FIELDS)).to_csv(cache, index=False)
        print(f"read {len(rows)} birth records -> {cache}")
    d = pd.read_csv(cache, dtype=str).apply(lambda c: c.str.strip())
    n0 = len(d)
    rf = ["RF_PDIAB", "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_PPTERM", "RF_INFTR"]
    d = d[(d.DPLURAL == "1") & d[rf].isin(["Y", "N"]).all(axis=1)]
    num = {c: pd.to_numeric(d[c], errors="coerce") for c in ["MAGER", "BMI", "CIG_0", "PRIORLIVE", "PRIORDEAD", "MEDUC", "PAY_REC"]}
    ok = (num["BMI"] < 99) & (num["CIG_0"] < 99) & (num["PRIORLIVE"] < 99) & (num["PRIORDEAD"] < 99) \
         & num["MEDUC"].between(1, 8) & (d.RF_PHYPE == "N")      # women with chronic hypertension excluded
    d = d[ok]; num = {k: v[ok] for k, v in num.items()}
    parity = num["PRIORLIVE"] + num["PRIORDEAD"]
    X = pd.DataFrame({
        "Age": num["MAGER"], "BMI": num["BMI"],
        "nulliparous": (parity == 0).astype(int), "prior_births": parity.clip(upper=6),
        "pregestational_diabetes": (d.RF_PDIAB == "Y").astype(int),
        "previous_preterm": (d.RF_PPTERM == "Y").astype(int), "infertility_treatment": (d.RF_INFTR == "Y").astype(int),
        "cigarettes_before": num["CIG_0"].clip(upper=40), "education": num["MEDUC"]})
    y = ((d.RF_GHYPE == "Y") | (d.RF_EHYPE == "Y")).astype(int)
    out = X.assign(y=y.values).sample(min(NAT_N, len(X)), random_state=SEED0).reset_index(drop=True)
    print(f"natality: {n0} records read | {len(X)} singleton births, no chronic hypertension, complete data | {len(out)} kept")
    return out

def load_raw():
    if DATASET == "natality":
        return load_natality()
    if DATASET == "uci":
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=863)
        X = d.data.features.copy()
        y_raw = d.data.targets.iloc[:, 0].astype(str).str.strip().str.lower()
        y = (y_raw == "high risk").astype(int)          # high risk vs low/mid risk
    else:
        df = pd.read_csv(CSV_PATH).drop(columns=DROP_COLS, errors="ignore")
        tcol = TARGET_COL or df.columns[-1]
        pos = POSITIVE if POSITIVE is not None else df[tcol].value_counts().idxmin()
        y = (df[tcol] == pos).astype(int); X = df.drop(columns=[tcol])
        print(f"Outcome column {tcol!r}, positive value {pos!r}")
    X = X.select_dtypes(include="number")
    X = X[X.columns[X.isna().mean() <= 0.30]]
    return X.assign(y=y.values)

def audit(df):
    if DATASET == "natality":   # registry data: identical rows are different women, so no de-duplication
        feats = [c for c in df.columns if c != "y"]
        print(f"features: {feats}\nprevalence of gestational hypertension/eclampsia: {df.y.mean():.3f}")
        return df.reset_index(drop=True)
    n0 = len(df); df = df.dropna(); n1 = len(df)
    df = df.drop_duplicates(); n2 = len(df)
    feats = [c for c in df.columns if c != "y"]
    conflict = df.groupby(feats)["y"].transform("nunique") > 1
    df = df[~conflict].reset_index(drop=True)
    print(f"rows: raw {n0} | complete {n1} | after removing exact duplicates {n2} "
          f"| after removing {int(conflict.sum())} conflicting rows {len(df)}")
    print(f"features: {feats}\nprevalence: {df.y.mean():.3f}")
    assert SHIFT_VAR in feats, f"SHIFT_VAR {SHIFT_VAR!r} is not a column"
    return df



def load_cdc():
    cache = "cdc_diabetes_binary.csv"
    if os.path.exists(cache):
        df = pd.read_csv(cache)
    else:
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=891)
        df = d.data.features.copy(); df["y"] = d.data.targets["Diabetes_binary"].values
        df.to_csv(cache, index=False)
    df = df.astype(float); df["y"] = df["y"].astype(int)
    print(f"CDC Diabetes Health Indicators: {len(df)} adults, prevalence {df.y.mean():.3f}")
    return df.reset_index(drop=True)

DATA = load_cdc() if DATASET == "cdc_diabetes" else audit(load_raw())
FEATURES = [c for c in DATA.columns if c != "y"]
assert SHIFT_VAR in FEATURES and AGE_VAR in FEATURES

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.optimize import brentq

X_ALL = DATA[FEATURES].to_numpy(float)
_Z = (X_ALL - X_ALL.mean(0)) / X_ALL.std(0)
_Z = np.nan_to_num(_Z)
_BETA = LogisticRegression(C=1e4, max_iter=5000).fit(_Z, DATA.y).coef_[0]
_zb = np.clip(_Z[:, FEATURES.index(SHIFT_VAR)], -2.5, 2.5)
_G = (_zb ** 2 - np.mean(_zb ** 2)) + _zb * _Z[:, FEATURES.index(AGE_VAR)]
_LIN = _Z @ _BETA
PREV = DATA.y.mean()
expit = lambda u: 1 / (1 + np.exp(-u))
ETA = {}
for g in GAMMAS:
    b0 = brentq(lambda c: expit(c + _LIN + g * _G).mean() - PREV, -30, 30)
    ETA[g] = expit(b0 + _LIN + g * _G)
_HIGH = (DATA[SHIFT_VAR] > DATA[SHIFT_VAR].median()).to_numpy()
RULE = (FEATURES.index(SHIFT_VAR), DATA[SHIFT_VAR].median(), False)

# Where does the real outcome sit on the misspecification axis? (held-out log-loss of logistic vs XGBoost on the real outcome)
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
_i = np.random.default_rng(0).permutation(len(DATA)); _tr, _te = _i[:40000], _i[40000:80000]
_lr = LogisticRegression(C=1.0, max_iter=5000).fit(_Z[_tr], DATA.y.iloc[_tr])
_xg = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, verbosity=0, n_jobs=-1).fit(_Z[_tr], DATA.y.iloc[_tr])
REAL_GAP = log_loss(DATA.y.iloc[_te], _lr.predict_proba(_Z[_te])[:, 1]) - log_loss(DATA.y.iloc[_te], _xg.predict_proba(_Z[_te])[:, 1])
print(f"real outcome: log-loss(logistic) - log-loss(XGBoost) = {REAL_GAP:+.4f} nats (rough misspecification index)")

natality: 72168 records read | 64783 singleton births, no chronic hypertension, complete data | 64783 kept
features: ['Age', 'BMI', 'nulliparous', 'prior_births', 'pregestational_diabetes', 'previous_preterm', 'infertility_treatment', 'cigarettes_before', 'education']
prevalence of gestational hypertension/eclampsia: 0.104
real outcome: log-loss(logistic) - log-loss(XGBoost) = -0.0004 nats (rough misspecification index)


## 3. Helpers (as in v8)

In [4]:
import torch, statsmodels.api as sm
from scipy import stats
from sklearn.tree import DecisionTreeRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

COLS = FEATURES + ["y"]

def draw(avail, n, share_high, rng):
    nh = int(round(share_high * n))
    idx = np.r_[rng.choice(np.flatnonzero(avail & _HIGH), nh, replace=False),
                rng.choice(np.flatnonzero(avail & ~_HIGH), n - nh, replace=False)]
    avail[idx] = False
    return rng.permutation(idx)                     # shuffle: draws are built high-then-low, and calibration CV folds must not follow that order

def cart_synth(df, n, seed):
    rng = np.random.default_rng(seed)
    out = pd.DataFrame({COLS[0]: rng.choice(df[COLS[0]].to_numpy(), n, replace=True)})
    for j, c in enumerate(COLS[1:], 1):
        prev = COLS[:j]
        tree = DecisionTreeRegressor(min_samples_leaf=CART_MIN_LEAF, random_state=seed + j).fit(df[prev], df[c])
        lr_, ls_ = tree.apply(df[prev]), tree.apply(out[prev])
        vals = df[c].to_numpy(); new = np.empty(n)
        groups = pd.Series(np.arange(len(df))).groupby(lr_).apply(np.array)
        for leaf in np.unique(ls_):
            pos = np.where(ls_ == leaf)[0]
            new[pos] = vals[rng.choice(groups[leaf], len(pos), replace=True)]
        out[c] = new
    return out

def _tilt_features(X):
    # features for the density-ratio classifier: all covariates plus squares of the non-binary ones
    nb = [j for j in range(X.shape[1]) if len(np.unique(X_ALL[:2000, j])) > 2]
    return np.c_[X, X[:, nb] ** 2]

def target_resample(pool_X, local_X, n, rng):
    # Targeting done by the RECEIVER on the synthetic pool it received (its own covariates never leave the site):
    # a probabilistic classifier separates local covariates from the pool, and its odds estimate the density ratio p_R / q
    F_pool, F_loc = _tilt_features(pool_X), _tilt_features(local_X)
    clf = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced"))])
    clf.fit(np.vstack([F_loc, F_pool]), np.r_[np.ones(len(F_loc)), np.zeros(len(F_pool))])
    p = np.clip(clf.predict_proba(F_pool)[:, 1], 1e-6, 1 - 1e-6)
    w = p / (1 - p); w = np.minimum(w, np.quantile(w, W_CLIP_Q)); w /= w.sum()
    return rng.choice(len(pool_X), size=n, replace=True, p=w), 1 / np.sum(w ** 2) / len(w)



# ---------------- v8 additions ----------------
T_NB = [0.10, 0.15, 0.20]          # risk thresholds for net benefit
B_CT = 30                          # simulations of the closed test with n local patients (finite-sample prediction)

def ratio_classifier(pool_X, local_X):
    # receiver-side density-ratio model: local covariates (1) vs the donor's synthetic covariates (0)
    F_pool, F_loc = _tilt_features(pool_X), _tilt_features(local_X)
    clf = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced"))])
    return clf.fit(np.vstack([F_loc, F_pool]), np.r_[np.ones(len(F_loc)), np.zeros(len(F_pool))])

def est_weights(clf, X):
    p = np.clip(clf.predict_proba(_tilt_features(X))[:, 1], 1e-6, 1 - 1e-6)
    w = p / (1 - p); w = np.minimum(w, np.quantile(w, W_CLIP_Q)); return w / w.mean()

def exact_weights(X, sk, rule):
    # the true density ratio p_R / p_donor of the simulated sites (known by design): sk/(1-sk) above the median, (1-sk)/sk below
    j, thr, ge = rule
    h = X[:, j] >= thr if ge else X[:, j] > thr
    return np.where(h, sk / (1 - sk), (1 - sk) / sk)

def resample_w(w, n, rng):
    p = w / w.sum(); return rng.choice(len(w), size=n, replace=True, p=p), 1 / np.sum(p ** 2) / len(p)

def fit_lr_w(X, y, w):
    m = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=1.0, max_iter=5000))])
    return m.fit(X, y, lr__sample_weight=w)

def net_benefit(y, p, t):
    pos = p >= t
    return float(np.mean(pos & (y == 1)) - np.mean(pos & (y == 0)) * t / (1 - t))

def cal_stats(y, p):
    lp = logit(p)
    s = _glm(y, sm.add_constant(lp, has_constant="add")); c = _glm(y, np.ones((len(y), 1)), offset=lp)
    return (np.nan if s is None else s.params[1]), (np.nan if c is None else c.params[0])

def ct_finite(lp_fn, X_pop, eta_pop, lp_te, e_te, n, B, rng):
    # expected excess risk of "population donor model + closed test on n local patients": the finite-sample
    # counterpart of Gamma_upd (the asymptotic version assumes perfect local updating)
    out = []
    for _ in range(B):
        i = rng.choice(len(X_pop), n, replace=False)
        yb = (rng.random(n) < eta_pop[i]).astype(int)
        p, _ = closed_test(lp_fn(X_pop[i]), yb, lp_te)
        out.append(kl_excess(e_te, p))
    return float(np.mean(out))

def fit_lr(X_hard=None, y_hard=None, X_soft=None, p_soft=None, C=1.0):
    # logistic regression on hard local labels and/or soft (teacher-probability) labels: each soft row is
    # duplicated with y=1 (weight p) and y=0 (weight 1-p), which is the infinite-relabelling limit of method B
    Xs, ys, ws = [], [], []
    if X_hard is not None:
        Xs.append(X_hard); ys.append(y_hard); ws.append(np.ones(len(y_hard)))
    if X_soft is not None:
        Xs += [X_soft, X_soft]; ys += [np.ones(len(p_soft)), np.zeros(len(p_soft))]; ws += [p_soft, 1 - p_soft]
    m = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(C=C, max_iter=5000))])
    return m.fit(np.vstack(Xs), np.concatenate(ys), lr__sample_weight=np.concatenate(ws))

def logit(p):
    p = np.clip(p, 1e-9, 1 - 1e-9); return np.log(p / (1 - p))

def _glm(y, X, offset=None, w=None):
    try:
        r = sm.GLM(y, X, family=sm.families.Binomial(), offset=offset, freq_weights=w).fit()
        if not r.converged or not np.all(np.isfinite(r.params)): return None
        return r
    except Exception:
        return None

def recal(lp_tr, y_tr, lp_te):
    r = _glm(y_tr, sm.add_constant(lp_tr, has_constant="add"))
    return expit(lp_te) if r is None else expit(r.params[0] + r.params[1] * lp_te)

def closed_test(lp_tr, y_tr, lp_te):
    # Closed testing (Vergouwe et al. 2017) among original / recalibration-in-the-large / logistic recalibration.
    # Model revision (re-estimating all coefficients) is not offered: with a few hundred patients it overfits badly.
    ll0 = np.sum(y_tr * np.log(expit(lp_tr)) + (1 - y_tr) * np.log(1 - expit(lp_tr)))
    m1 = _glm(y_tr, np.ones((len(y_tr), 1)), offset=lp_tr)
    m2 = _glm(y_tr, sm.add_constant(lp_tr, has_constant="add"))
    ll = lambda m: -np.inf if m is None else m.llf
    sig = lambda a, b, df: 2 * (ll(a) - b) > stats.chi2.ppf(1 - ALPHA_CT, df)
    if m2 is None or not sig(m2, ll0, 2):
        return expit(lp_te), "original"
    if sig(m2, ll(m1), 1):
        return expit(m2.params[0] + m2.params[1] * lp_te), "recalibration"
    return (expit(m1.params[0] + lp_te), "intercept") if m1 is not None else (expit(lp_te), "original")

def kl_excess(eta, p):
    p = np.clip(p, 1e-9, 1 - 1e-9); e = np.clip(eta, 1e-12, 1 - 1e-12)
    return float(np.mean(e * np.log(e / p) + (1 - e) * np.log((1 - e) / (1 - p))))

def teacher_fit(X, y, seed):
    xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9,
                        eval_metric="logloss", random_state=seed, n_jobs=-1, verbosity=0)
    return CalibratedClassifierCV(xgb, method="sigmoid", cv=5).fit(X, y)

# ---------------- v11c: COMMUTE-style transfer (per-donor calibration + synthetic data at the receiver) ----------------
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
LAMBDAS = [0.3, 1, 3, 10, 30, 100, 300]

def _fit_delta(off, Z, y, lam):
    # logit p = (shared linear predictor) + d0 + Z @ d,  ridge penalty lam/2 * ||d||^2 (distance to the shared model)
    def f(w):
        eta = off + w[0] + Z @ w[1:]
        g = expit(eta) - y
        return (np.sum(np.logaddexp(0, eta) - y * eta) + 0.5 * lam * np.sum(w[1:] ** 2),
                np.r_[g.sum(), Z.T @ g + lam * w[1:]])
    return minimize(f, np.zeros(Z.shape[1] + 1), jac=True, method="L-BFGS-B").x

def transfer_fit(lp_loc, X_loc, y_loc, seed):
    sc = StandardScaler().fit(X_loc); Zl = np.nan_to_num(sc.transform(X_loc))
    if min(y_loc.sum(), len(y_loc) - y_loc.sum()) < 5:
        lam = max(LAMBDAS)
    else:
        cv = StratifiedKFold(5, shuffle=True, random_state=seed); losses = []
        for lam_ in LAMBDAS:
            L = 0.0
            for tr, va in cv.split(Zl, y_loc):
                w = _fit_delta(lp_loc[tr], Zl[tr], y_loc[tr], lam_)
                p = np.clip(expit(lp_loc[va] + w[0] + Zl[va] @ w[1:]), 1e-9, 1 - 1e-9)
                L -= np.sum(y_loc[va] * np.log(p) + (1 - y_loc[va]) * np.log(1 - p))
            losses.append(L)
        lam = LAMBDAS[int(np.argmin(losses))]
    return sc, _fit_delta(lp_loc, Zl, y_loc, lam)

def transfer_predict(tf, lp, X_new):
    sc, w = tf; Z = np.nan_to_num(sc.transform(X_new))
    return expit(lp + w[0] + Z @ w[1:])

def transfer_lr(lp_loc, X_loc, y_loc, lp_te, X_te, seed):
    return transfer_predict(transfer_fit(lp_loc, X_loc, y_loc, seed), lp_te, X_te)


## 4. Run (resumes from the checkpoint file)

In [5]:
COLUMNS = ["dataset", "gamma", "share", "config", "rep", "arm", "ex_kl", "brier_y", "auc", "cal_slope", "nb20", "ct_choice", "ess_C", "ess_IW"]

def done_keys():
    if not os.path.exists(OUT_CSV): return set()
    r = pd.read_csv(OUT_CSV); return set(zip(r.gamma, r["share"], r.config, r.rep))

def run_cell(gam, sk, cfg_name, rep):
    t0 = time.time()
    seed = SEED0 + 1000 * rep + 100 * GAMMAS.index(gam) + 10 * SKEWS.index(sk) + CFG_NAMES.index(cfg_name)
    rng = np.random.default_rng(seed)
    eta = ETA[gam]; X = X_ALL
    avail = np.ones(len(DATA), bool)
    te, loc, rpop = draw(avail, TEST_N, sk, rng), draw(avail, N_LOCAL, sk, rng), draw(avail, POP_N, sk, rng)
    donors = [draw(avail, n, dshare(k, sk), rng) for k, n in CONFIGS[cfg_name]]
    dpops = [draw(avail, POP_N, dshare(k, sk), rng) for k, _ in CONFIGS[cfg_name]]
    y = (rng.random(len(DATA)) < eta).astype(int)
    e_te = eta[te]
    res, extra = {}, {}

    res["local"] = fit_lr(X[loc], y[loc]).predict_proba(X[te])[:, 1]
    h_star = fit_lr(X_soft=X[rpop], p_soft=eta[rpop], C=1e4)
    res["oracle_h_star"] = h_star.predict_proba(X[te])[:, 1]
    dstars = [fit_lr(X_soft=X[dp], p_soft=eta[dp], C=1e4) for dp in dpops]
    lp_mix = lambda Z: np.mean([m.decision_function(Z) for m in dstars], 0)
    res["oracle_mix"] = expit(lp_mix(X[te]))
    extra["oracle_mix_ct_n"] = ct_finite(lp_mix, X[rpop], eta[rpop], lp_mix(X[te]), e_te, N_LOCAL, B_CT, rng)

    # model sharing: every donor publishes its logistic score; the receiver averages the logits, then closed testing
    hDs = [fit_lr(X[d], y[d]) for d in donors]
    lpm = lambda Z: np.mean([m.decision_function(Z) for m in hDs], 0)
    res["share_mix"] = expit(lpm(X[te]))
    res["share_mix_ct"], ct = closed_test(lpm(X[loc]), y[loc], lpm(X[te]))
    # COMMUTE-style transfer (one exchange of models; everything else happens at the receiver)
    res["transfer_mix"] = transfer_lr(lpm(X[loc]), X[loc], y[loc], lpm(X[te]), X[te], seed)       # single transfer of the averaged score
    tfs = [transfer_fit(m.decision_function(X[loc]), X[loc], y[loc], seed + 50 + k) for k, m in enumerate(hDs)]
    ldf = pd.DataFrame(X[loc], columns=FEATURES).assign(y=y[loc])
    Xs = cart_synth(ldf, N_SYN, seed + 77)[FEATURES].to_numpy(float)                             # synthetic covariates from the receiver's own patients
    ps = np.mean([transfer_predict(tf, m.decision_function(Xs), Xs) for tf, m in zip(tfs, hDs)], 0)  # labelled by the calibrated donor models
    res["COMMUTE_like"] = fit_lr(X[loc], y[loc], Xs, ps).predict_proba(X[te])[:, 1]

    # each donor: teacher, synthetic pool, and a reweighted refit (one exchange per donor)
    pools, ppools, iw, ess_iw = [], [], [], []
    for k, d in enumerate(donors):
        t = teacher_fit(X[d], y[d], seed + k)
        ddf = pd.DataFrame(X[d], columns=FEATURES).assign(y=y[d])
        pool = cart_synth(ddf, POOL_MULT * len(d), seed + k)[FEATURES].to_numpy(float)
        pools.append(pool); ppools.append(t.predict_proba(pool)[:, 1])
        clf_k = ratio_classifier(pool, X[loc]); w_k = est_weights(clf_k, X[d])
        iw.append(fit_lr_w(X[d], y[d], w_k)); ess_iw.append(w_k.sum() ** 2 / np.sum(w_k ** 2) / len(w_k))
    lpiw = lambda Z: np.mean([m.decision_function(Z) for m in iw], 0)
    res["IW_mix"] = expit(lpiw(X[te]))

    # targeted synthetic data on the union of all pools
    P, pp = np.vstack(pools), np.concatenate(ppools)
    clf = ratio_classifier(P, X[loc])
    idxC, ess = resample_w(est_weights(clf, P), N_SYN, rng)
    res["C_union"] = fit_lr(X[loc], y[loc], P[idxC], pp[idxC]).predict_proba(X[te])[:, 1]
    B = rng.choice(len(P), N_SYN, replace=False)
    res["B_union"] = fit_lr(X[loc], y[loc], P[B], pp[B]).predict_proba(X[te])[:, 1]

    rows = []
    for arm, p in res.items():
        pc = np.clip(p, 1e-9, 1 - 1e-9)
        rows.append(dict(dataset=DATASET, gamma=gam, share=sk, config=cfg_name, rep=rep, arm=arm, ex_kl=kl_excess(e_te, pc),
                         brier_y=float(np.mean((pc - y[te]) ** 2)), auc=roc_auc_score(y[te], pc),
                         cal_slope=cal_stats(y[te], pc)[0], nb20=net_benefit(y[te], pc, 0.20),
                         ct_choice=ct if arm == "share_mix_ct" else "", ess_C=ess if arm == "C_union" else np.nan,
                         ess_IW=float(np.mean(ess_iw)) if arm == "IW_mix" else np.nan))
    for arm, v in extra.items():
        rows.append(dict(dataset=DATASET, gamma=gam, share=sk, config=cfg_name, rep=rep, arm=arm, ex_kl=v))
    pd.DataFrame(rows).reindex(columns=COLUMNS).to_csv(OUT_CSV, mode="a", header=not os.path.exists(OUT_CSV), index=False)
    print(f"  gamma {gam:.0f} share {sk:.1f} {cfg_name:15s} rep {rep:2d}  {time.time() - t0:5.1f}s")

t_start = time.time()
for rep, gam, sk, cfg_name in itertools.product(range(N_REPS), GAMMAS, SKEWS, CFG_NAMES):
    if (gam, sk, cfg_name, rep) not in done_keys(): run_cell(gam, sk, cfg_name, rep)
print(f"Finished in {(time.time() - t_start) / 60:.1f} min -> {OUT_CSV}")

  gamma 0 share 0.7 single          rep  0    3.1s
  gamma 0 share 0.7 three_opposite  rep  0    5.5s
  gamma 0 share 0.7 three_diverse   rep  0    5.7s
  gamma 0 share 0.9 single          rep  0    2.5s
  gamma 0 share 0.9 three_opposite  rep  0    5.3s
  gamma 0 share 0.9 three_diverse   rep  0    5.4s
  gamma 1 share 0.7 single          rep  0    2.4s
  gamma 1 share 0.7 three_opposite  rep  0    5.4s
  gamma 1 share 0.7 three_diverse   rep  0    5.4s
  gamma 1 share 0.9 single          rep  0    2.5s
  gamma 1 share 0.9 three_opposite  rep  0    5.5s
  gamma 1 share 0.9 three_diverse   rep  0    5.3s
  gamma 0 share 0.7 single          rep  1    2.4s
  gamma 0 share 0.7 three_opposite  rep  1    5.3s
  gamma 0 share 0.7 three_diverse   rep  1    5.3s
  gamma 0 share 0.9 single          rep  1    2.5s
  gamma 0 share 0.9 three_opposite  rep  1    5.4s
  gamma 0 share 0.9 three_diverse   rep  1    5.4s
  gamma 1 share 0.7 single          rep  1    2.5s
  gamma 1 share 0.7 three_oppos

## 5. Results and hypotheses

In [6]:
R = pd.read_csv(OUT_CSV)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)
CELL = ["gamma", "share", "config"]
def gain(sub, a, b, m="ex_kl"):
    A = sub[sub.arm == a].set_index("rep")[m]; B = sub[sub.arm == b].set_index("rep")[m]
    d = (A - B).dropna() * 1000
    if len(d) < 2: return d.mean(), np.nan, np.nan
    h = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))
    return d.mean(), d.mean() - h, d.mean() + h
f = lambda g: f"{g[0]:+.1f} [{g[1]:+.1f}, {g[2]:+.1f}]"
rows = []
for key, sub in R.groupby(CELL):
    pr = sub.groupby("arm").ex_kl.mean() * 1000
    rows.append(dict(zip(CELL, key), Gamma_mix=round(pr["oracle_mix"] - pr["oracle_h_star"], 1),
                     Gamma_mix_n=round(pr["oracle_mix_ct_n"] - pr["oracle_h_star"], 1),
                     share_ct=round(pr["share_mix_ct"], 1),
                     IW=f(gain(sub, "share_mix_ct", "IW_mix")), C=f(gain(sub, "share_mix_ct", "C_union")),
                     B=f(gain(sub, "share_mix_ct", "B_union")), C_minus_IW=f(gain(sub, "IW_mix", "C_union")),
                     ESS_C=round(sub[sub.arm == "C_union"].ess_C.mean(), 3), ESS_IW=round(sub[sub.arm == "IW_mix"].ess_IW.mean(), 3)))
T = pd.DataFrame(rows)
print("Gains over updated model sharing (averaged donor scores + closed testing), excess log-loss x 1000; positive = arm better")
display(T)

G = lambda g, s, c, a, b: gain(R[(R.gamma == g) & (R["share"] == s) & (R.config == c)], a, b)
row = lambda g, s, c: T[(T.gamma == g) & (T["share"] == s) & (T.config == c)].iloc[0]
M1a = all(G(0.0, s, c, "share_mix_ct", "C_union")[1] <= 0 for s in SKEWS for c in CFG_NAMES)
M1b = all(G(1.0, s, c, "share_mix_ct", "C_union")[1] > 0 for s in SKEWS for c in CFG_NAMES if row(1.0, s, c).Gamma_mix_n >= 2)
M2 = all(row(g, s, "three_diverse").ESS_C > row(g, s, "single").ESS_C for g in GAMMAS for s in SKEWS)
M3 = all(row(1.0, s, "three_diverse").Gamma_mix < row(1.0, s, "single").Gamma_mix and
         G(1.0, s, "three_diverse", "share_mix_ct", "C_union")[0] < G(1.0, s, "single", "share_mix_ct", "C_union")[0] for s in SKEWS)
M4 = all(G(1.0, s, "three_opposite", "IW_mix", "C_union")[0] > G(1.0, s, "single", "IW_mix", "C_union")[0] for s in SKEWS)
print(f"\nM1 (no gain at gamma=0): {M1a} | (gain where predicted at gamma=1): {M1b}\nM2 (diverse union -> larger ESS): {M2}\n"
      f"M3 (diverse donors -> smaller predicted and observed gain): {M3}\nM4 (smaller mirrored donors -> larger synthetic advantage over reweighting): {M4}")
print("\nClinical metrics (gamma = 1)")
display(R[(R.gamma == 1.0) & R.arm.isin(["share_mix_ct", "IW_mix", "C_union"])].groupby(["share", "config", "arm"])[["brier_y", "auc", "cal_slope", "nb20"]].mean().round(4))

Gains over updated model sharing (averaged donor scores + closed testing), excess log-loss x 1000; positive = arm better


,gamma,share,config,Gamma_mix,Gamma_mix_n,share_ct,IW,C,B,C_minus_IW,ESS_C,ESS_IW
0,0.0,0.7,single,-0.0,1.0,1.6,"-0.2 [-2.3, +1.9]","-2.8 [-5.0, -0.7]","-3.4 [-5.6, -1.2]","-2.7 [-3.7, -1.6]",0.597,0.597
1,0.0,0.7,three_diverse,-0.0,1.1,0.6,"-0.2 [-0.3, -0.1]","-5.8 [-6.3, -5.2]","-6.0 [-6.5, -5.5]","-5.6 [-6.2, -5.0]",0.842,0.794
2,0.0,0.7,three_opposite,-0.0,1.1,1.5,"-0.2 [-1.6, +1.3]","-7.3 [-9.5, -5.1]","-7.8 [-10.0, -5.7]","-7.1 [-8.3, -5.9]",0.607,0.606
3,0.0,0.9,single,0.0,1.4,4.3,"-4.2 [-10.2, +1.8]","-4.2 [-9.7, +1.2]","-3.7 [-9.2, +1.7]","-0.0 [-2.2, +2.1]",0.083,0.083
4,0.0,0.9,three_diverse,0.0,1.0,1.3,"-1.8 [-3.1, -0.5]","-4.6 [-6.1, -3.0]","-5.1 [-6.5, -3.7]","-2.7 [-4.0, -1.4]",0.585,0.537
5,0.0,0.9,three_opposite,0.0,1.0,1.4,"-9.5 [-11.3, -7.6]","-13.7 [-15.6, -11.9]","-13.0 [-14.5, -11.5]","-4.3 [-7.0, -1.5]",0.083,0.083
6,1.0,0.7,single,28.9,30.6,98.5,"+28.3 [+23.9, +32.7]","+30.3 [+26.3, +34.4]","+8.7 [+5.0, +12.3]","+2.0 [+0.3, +3.7]",0.593,0.594
7,1.0,0.7,three_diverse,6.4,10.7,76.5,"+9.0 [+6.6, +11.4]","+7.3 [+4.9, +9.8]","-1.6 [-4.2, +1.0]","-1.7 [-2.2, -1.1]",0.844,0.792
8,1.0,0.7,three_opposite,27.3,29.6,97.0,"+26.3 [+23.8, +28.8]","+27.0 [+24.6, +29.4]","+5.3 [+3.6, +6.9]","+0.6 [-0.4, +1.7]",0.590,0.589
9,1.0,0.9,single,143.6,142.8,174.0,"+121.3 [+113.4, +129.3]","+130.8 [+125.4, +136.1]","+38.4 [+32.8, +43.9]","+9.4 [+5.1, +13.7]",0.088,0.088



M1 (no gain at gamma=0): True | (gain where predicted at gamma=1): True
M2 (diverse union -> larger ESS): True
M3 (diverse donors -> smaller predicted and observed gain): True
M4 (smaller mirrored donors -> larger synthetic advantage over reweighting): False

Clinical metrics (gamma = 1)


brier_y     auc  cal_slope    nb20
share config         arm                                             
0.7   single         C_union        0.0626  0.8421     0.9794  0.0722
                     IW_mix         0.0618  0.8422     0.8701  0.0716
                     share_mix_ct   0.0811  0.8219     1.2396  0.0548
      three_diverse  C_union        0.0650  0.8406     1.1576  0.0718
                     IW_mix         0.0622  0.8408     0.9544  0.0710
                     share_mix_ct   0.0682  0.8374     1.0517  0.0679
      three_opposite C_union        0.0639  0.8386     1.0644  0.0723
                     IW_mix         0.0614  0.8390     0.8751  0.0714
                     share_mix_ct   0.0805  0.8191     1.1935  0.0548
0.9   single         C_union        0.0608  0.8956     0.8709  0.0943
                     IW_mix         0.0619  0.8937     0.7457  0.0933
                     share_mix_ct   0.1130  0.7263     0.8850  0.0291
      three_diverse  C_union        0.0631  0.8991     1.1694  0.0966
                     IW_mix         0.0612  0.8968     0.8716  0.0958
                     share_mix_ct   0.0730  0.8867     1.1308  0.0895
      three_opposite C_union        0.0627  0.8990     1.0560  0.0963
                     IW_mix         0.0615  0.8974     0.7461  0.0952
                     share_mix_ct   0.1151  0.7167     0.8482  0.0259

## Comparison with COMMUTE

In [7]:

R = pd.read_csv(OUT_CSV)
rows = []
for key, sub in R.groupby(CELL):
    rows.append(dict(zip(CELL, key), transfer_mix=f(gain(sub, "share_mix_ct", "transfer_mix")),
                     COMMUTE_like=f(gain(sub, "share_mix_ct", "COMMUTE_like")), IW=f(gain(sub, "share_mix_ct", "IW_mix")),
                     C_union=f(gain(sub, "share_mix_ct", "C_union")), C_minus_COMMUTE=f(gain(sub, "COMMUTE_like", "C_union")),
                     COMMUTE_minus_transfer=f(gain(sub, "transfer_mix", "COMMUTE_like"))))
TC = pd.DataFrame(rows)
print("COMMUTE comparison: gains over updated model sharing (x1000, positive = arm better)")
display(TC)
K1 = all(G(1.0, s, c, "COMMUTE_like", "C_union")[1] > 0 for s in SKEWS for c in CFG_NAMES)
K2 = all(G(g, s, c, "transfer_mix", "COMMUTE_like")[0] >= 0 for g in GAMMAS for s in SKEWS for c in CFG_NAMES)
K3 = all(G(0.0, s, c, "share_mix_ct", "COMMUTE_like")[1] <= 0 for s in SKEWS for c in CFG_NAMES)
print(f"\nK1 (targeted synthetic beats COMMUTE where gain predicted): {K1}\nK2 (COMMUTE >= single transfer): {K2}\nK3 (COMMUTE no gain at gamma=0): {K3}")
display(R[(R.gamma == 1.0) & R.arm.isin(["share_mix_ct", "COMMUTE_like", "IW_mix", "C_union"])].groupby(["share", "config", "arm"])[["brier_y", "auc", "cal_slope"]].mean().round(4))


COMMUTE comparison: gains over updated model sharing (x1000, positive = arm better)


,gamma,share,config,transfer_mix,COMMUTE_like,IW,C_union,C_minus_COMMUTE,COMMUTE_minus_transfer
0,0.0,0.7,single,"-6.9 [-12.9, -0.9]","-9.0 [-17.2, -0.8]","-0.2 [-2.3, +1.9]","-2.8 [-5.0, -0.7]","+6.2 [-2.0, +14.3]","-2.1 [-6.4, +2.1]"
1,0.0,0.7,three_diverse,"-5.9 [-9.6, -2.2]","-3.9 [-6.1, -1.8]","-0.2 [-0.3, -0.1]","-5.8 [-6.3, -5.2]","-1.9 [-4.0, +0.3]","+1.9 [-0.2, +4.1]"
2,0.0,0.7,three_opposite,"-4.9 [-9.0, -0.9]","-4.0 [-7.3, -0.6]","-0.2 [-1.6, +1.3]","-7.3 [-9.5, -5.1]","-3.3 [-6.4, -0.3]","+1.0 [-1.6, +3.6]"
3,0.0,0.9,single,"-2.5 [-5.3, +0.3]","-1.9 [-6.2, +2.3]","-4.2 [-10.2, +1.8]","-4.2 [-9.7, +1.2]","-2.3 [-5.3, +0.8]","+0.5 [-2.0, +3.0]"
4,0.0,0.9,three_diverse,"-6.4 [-11.0, -1.8]","-4.8 [-7.3, -2.4]","-1.8 [-3.1, -0.5]","-4.6 [-6.1, -3.0]","+0.3 [-2.3, +2.9]","+1.6 [-2.7, +5.8]"
5,0.0,0.9,three_opposite,"-3.5 [-5.2, -1.8]","-3.2 [-4.9, -1.5]","-9.5 [-11.3, -7.6]","-13.7 [-15.6, -11.9]","-10.5 [-13.8, -7.2]","+0.3 [-0.8, +1.5]"
6,1.0,0.7,single,"+10.7 [+5.5, +15.9]","+10.4 [+5.3, +15.6]","+28.3 [+23.9, +32.7]","+30.3 [+26.3, +34.4]","+19.9 [+12.3, +27.4]","-0.3 [-2.9, +2.4]"
7,1.0,0.7,three_diverse,"-1.1 [-5.3, +3.0]","+0.5 [-2.8, +3.7]","+9.0 [+6.6, +11.4]","+7.3 [+4.9, +9.8]","+6.9 [+3.2, +10.5]","+1.6 [-0.7, +3.9]"
8,1.0,0.7,three_opposite,"+6.2 [-1.2, +13.6]","+7.1 [-1.5, +15.6]","+26.3 [+23.8, +28.8]","+27.0 [+24.6, +29.4]","+19.9 [+12.5, +27.3]","+0.9 [-1.1, +2.8]"
9,1.0,0.9,single,"+101.5 [+93.7, +109.3]","+101.3 [+92.9, +109.8]","+121.3 [+113.4, +129.3]","+130.8 [+125.4, +136.1]","+29.4 [+22.0, +36.8]","-0.2 [-3.0, +2.7]"



K1 (targeted synthetic beats COMMUTE where gain predicted): True
K2 (COMMUTE >= single transfer): False
K3 (COMMUTE no gain at gamma=0): True


brier_y     auc  cal_slope
share config         arm                                     
0.7   single         COMMUTE_like   0.0719  0.8317     1.0464
                     C_union        0.0626  0.8421     0.9794
                     IW_mix         0.0618  0.8422     0.8701
                     share_mix_ct   0.0811  0.8219     1.2396
      three_diverse  COMMUTE_like   0.0675  0.8366     1.0661
                     C_union        0.0650  0.8406     1.1576
                     IW_mix         0.0622  0.8408     0.9544
                     share_mix_ct   0.0682  0.8374     1.0517
      three_opposite COMMUTE_like   0.0714  0.8255     0.9887
                     C_union        0.0639  0.8386     1.0644
                     IW_mix         0.0614  0.8390     0.8751
                     share_mix_ct   0.0805  0.8191     1.1935
0.9   single         COMMUTE_like   0.0744  0.8762     1.0084
                     C_union        0.0608  0.8956     0.8709
                     IW_mix         0.0619  0.8937     0.7457
                     share_mix_ct   0.1130  0.7263     0.8850
      three_diverse  COMMUTE_like   0.0679  0.8921     1.1093
                     C_union        0.0631  0.8991     1.1694
                     IW_mix         0.0612  0.8968     0.8716
                     share_mix_ct   0.0730  0.8867     1.1308
      three_opposite COMMUTE_like   0.0720  0.8831     1.0787
                     C_union        0.0627  0.8990     1.0560
                     IW_mix         0.0615  0.8974     0.7461
                     share_mix_ct   0.1151  0.7167     0.8482

## 6. What to send back
`results_v10_<dataset>_full.csv`